# 2 · Permeability from logs, tested blind: Zeerijp-3A

**Question.** In a well or interval without core, how well can permeability be predicted from logs, and how much do rock types help?

**Test.** *Leave one core out*: each of the cored intervals is hidden in turn, the methods are trained on the others, and their prediction for the hidden core is compared with its plugs. Three methods: (A) one trend of log k on log density porosity; (B) rock types predicted from logs by a random forest, then k from each type's FZI; (C) a random forest that predicts log k directly from the logs.

**Logs.** Density, neutron, photoelectric, gamma ray, induction resistivity and sonic of ZRP-3A (Schlumberger runs of 18 Aug 2015), from NLOG.

In [ ]:
import json, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio
import subsurface_tools as kt
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False, "savefig.bbox": "tight", "savefig.facecolor": "white"})
SAND, MUD, INK = "#d9a441", "#5b4237", "#1f2a36"
pathlib.Path("results").mkdir(exist_ok=True); pathlib.Path("figures").mkdir(exist_ok=True)
core_file = kt.fetch_nlog("core"); log_file = kt.fetch_nlog("logs"); sonic_file = kt.fetch_nlog("sonic")
plugs = kt.read_zrp_plugs(core_file)
print(len(plugs), "plug measurements from", plugs.sheet.nunique(), "sheets")
L = lasio.read(log_file, engine="normal").df().reset_index().sort_values("DEPTH")
S = lasio.read(sonic_file, engine="normal").df().reset_index()[["DEPTH", "MCDT"]].sort_values("DEPTH")
L = pd.merge_asof(L, S, on="DEPTH", direction="nearest", tolerance=0.06)
L.loc[(L.DPOR < -20) | (L.DPOR > 60), "DPOR"] = np.nan   # tool start-up values
feats = ["GRGC", "DEN", "NPOR", "DPOR", "RILD", "PDPE", "MCDT"]
print(L[["DEPTH"] + feats].describe().loc[["min", "max"]].round(2))

## Depth match
Core depths are driller's depths; logs are wireline depths. The shift that best lines up core porosity with log density porosity is searched in 0.1 m steps.

In [ ]:
h = plugs[(plugs.orient == "H") & plugs.k_kl.notna() & (plugs.k_kl > 0) & plugs.phi.notna() & ~plugs.damaged].drop_duplicates("depth").sort_values("depth").copy()
best = max(((round(s, 1), pd.merge_asof(h.assign(D=h.depth + s).sort_values("D"), L, left_on="D", right_on="DEPTH", direction="nearest", tolerance=0.06)
           .pipe(lambda m: m.phi.corr(m.DPOR))) for s in np.arange(-3, 3.01, 0.1)), key=lambda t: t[1])
print("best shift (m):", best[0], " correlation core phi vs log DPOR:", round(best[1], 3))
m = pd.merge_asof(h.assign(D=h.depth + best[0]).sort_values("D"), L, left_on="D", right_on="DEPTH", direction="nearest", tolerance=0.06).dropna(subset=feats)
m["RQI"], m["FZI"] = kt.fzi(m.phi, m.k_kl); m["RT"], _, _ = kt.rock_types(m.FZI, n=3); m["lk"] = np.log10(m.k_kl)
print(len(m), "plugs with a full log suite, cores", sorted(m.core.unique()))

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
rows = []
for c in sorted(m.core.unique()):
    tr, te = m[m.core != c], m[m.core == c]
    a = np.polyfit(tr.DPOR / 100, tr.lk, 1)
    clf = RandomForestClassifier(400, min_samples_leaf=3, random_state=0).fit(tr[feats], tr.RT)
    fzt = tr.groupby("RT").FZI.apply(lambda s: np.exp(np.log(s).mean()))
    rtp = clf.predict(te[feats])
    reg = RandomForestRegressor(400, min_samples_leaf=3, random_state=0).fit(tr[feats], tr.lk)
    rows.append(pd.DataFrame({"depth": te.depth, "core": c, "obs": te.lk, "rt": te.RT, "rt_pred": rtp,
        "A_trend": np.polyval(a, te.DPOR / 100),
        "B_rocktypes": np.log10(kt.k_from_fzi((te.DPOR / 100).clip(.02, .4), pd.Series(rtp, index=te.index).map(fzt))),
        "C_forest": reg.predict(te[feats])}))
B = pd.concat(rows)
blind = {k: {"within x3": round(kt.within(B[k], B.obs), 3), "within x10": round(kt.within(B[k], B.obs, 10), 3),
             "rmse log10 k": round(float(np.sqrt(np.mean((B[k] - B.obs) ** 2))), 3)} for k in ["A_trend", "B_rocktypes", "C_forest"]}
blind["rock type from logs: accuracy"] = round(float((B.rt == B.rt_pred).mean()), 3)
print(json.dumps(blind, indent=1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for k, c in [("A_trend", "#9aa7b4"), ("C_forest", SAND)]:
    ax[0].plot(B.obs, B[k], ".", ms=3, alpha=.6, color=c, label=f"{k}: {blind[k]['within x3']:.0%} within x3")
ax[0].plot([-3, 4], [-3, 4], "k-", lw=.8); ax[0].set_xlabel("measured log10 k (hidden core)"); ax[0].set_ylabel("predicted from logs"); ax[0].legend(fontsize=8)
ax[0].set_title("Blind test, one core hidden at a time")
w = pd.Series({k: blind[k]["within x3"] for k in ["A_trend", "B_rocktypes", "C_forest"]})
ax[1].bar(["one trend", "rock types\nfrom logs", "random forest"], w.values, color=["#9aa7b4", "#b07a3b", SAND])
ax[1].set_ylim(0, 1); ax[1].set_ylabel("share of plugs within a factor of 3"); ax[1].set_title("Logs only, uncored case")
fig.tight_layout(); fig.savefig("figures/zrp_blind_test.png")

## The deliverable: a permeability log with its uncertainty
Trained on all cores, the forest predicts k every 0.1 m over the logged interval. The band is the spread of the individual trees, widened to match the blind-test error.

In [ ]:
reg = RandomForestRegressor(400, min_samples_leaf=3, random_state=0).fit(m[feats], m.lk)
W = L.dropna(subset=feats).copy(); trees = np.stack([t.predict(W[feats].values) for t in reg.estimators_])
W["k50"] = 10 ** np.median(trees, 0); spread = trees.std(0)
lo, hi = m[feats].quantile(.01), m[feats].quantile(.99)
W["inside"] = ((W[feats] >= lo) & (W[feats] <= hi)).all(axis=1)   # log responses seen in the cored rock
scale = blind["C_forest"]["rmse log10 k"] / float(np.sqrt(np.mean(spread ** 2)))
W["k10"] = 10 ** (np.log10(W.k50) - 1.28 * spread * scale); W["k90"] = 10 ** (np.log10(W.k50) + 1.28 * spread * scale)
fig, ax = plt.subplots(1, 3, figsize=(10, 8), sharey=True)
ax[0].plot(W.GRGC, W.DEPTH, lw=.6, color="#4c7a3d"); ax[0].set_xlabel("GR (API)"); ax[0].invert_yaxis(); ax[0].set_ylabel("depth (m)")
ax[1].plot(W.DPOR / 100, W.DEPTH, lw=.6, color=INK, label="log"); ax[1].plot(h.phi, h.depth + best[0], "o", ms=2, color=SAND, label="core"); ax[1].set_xlabel("porosity"); ax[1].legend(fontsize=7)
Wi = W.where(W.inside); Wo = W.where(~W.inside)
ax[2].fill_betweenx(W.DEPTH, Wi.k10, Wi.k90, color=SAND, alpha=.35, lw=0, label="P10-P90"); ax[2].semilogx(Wi.k50, W.DEPTH, lw=.7, color="#8a5a1f", label="P50 from logs")
ax[2].semilogx(Wo.k50, W.DEPTH, lw=.7, color="0.7", label="outside the cored rock (not trusted)")
ax[2].semilogx(h.k_kl, h.depth + best[0], ".", ms=2, color="k", label="core plugs"); ax[2].set_xlabel("k (mD)"); ax[2].legend(fontsize=7)
fig.suptitle("ZRP-3A: permeability from logs with uncertainty"); fig.tight_layout(); fig.savefig("figures/zrp_k_log.png")
print(f"{W.inside.mean():.0%} of the logged interval has log responses inside the range of the cored rock")
W[["DEPTH", "k10", "k50", "k90", "inside"]].round(4).to_csv("results/zrp3a_k_from_logs.csv", index=False)

In [ ]:
json.dump({"depth_shift_m": best[0], "core_log_phi_corr": round(best[1], 3), "blind_test": blind, "plugs": int(len(m))},
          open("results/nb2_blind_test.json", "w"), indent=1)
assert blind["C_forest"]["within x3"] >= blind["A_trend"]["within x3"] - 0.05

## Result and limits
From logs alone, about 6 in 10 plugs of a hidden core are predicted within a factor of 3. Rock types predicted from logs do not beat the simpler methods here: the logs (about 0.5 m vertical resolution) cannot see the centimetre laminae that separate the types. That is the honest limit to state to a client, and the reason the product reports P10–P90, not one number. The gain from rock typing is real where there is core (notebook 1).